In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

import gc
import itertools


In [25]:
base_path = "/mnt/SingleTrackTrees/Data/alice_data_2023_LHC23f_535087_apass4_1300_Thinner/Output/Run535087/"
which_chunk = "Chunk0950"
which_number = "001_007"
which_file = "AO2Dtree.root"
path = base_path + "".join([which_chunk, which_number, which_file])
#path = "~/Downloads/AO2Dtree.root"
file = uproot.open(path)

In [3]:
# NSigmaTPC:
sigma_limit = 3
cut1 = (
    "( (fNsigmaTPCpi > -3) & (fNsigmaTPCpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTPCka > -3) & (fNsigmaTPCka < 3) & (fCharge == -1) ) | "
    "( (fPt > 0) & (fPt < 1) & (fNsigmaTPCka > -15) & (fNsigmaTPCka < 0) & (fCharge == 1) )"
)

# NsigmaTOF:
sigma_limit = 3
cut2 = (
    "( (fNsigmaTOFpi > -3) & (fNsigmaTOFpi < 3) & (fCharge == 1) ) | "
    "( (fNsigmaTOFka > -3) & (fNsigmaTOFka < 3) & (fCharge == -1) ) | "
    "( (fNsigmaTOFka > 998.5) | (fNsigmaTOFka < -998.5) | (fNsigmaTOFpi > 998.5) | (fNsigmaTOFpi < -998.5) ) | "
    "( (fPt > 0) & (fPt < 3) & (fNsigmaTOFka > -50) & (fNsigmaTOFka < 0) & (fCharge == 1) )"
)

# DCA_XY:
cut3 = "( (fDcaXY > 0.0002) | (fDcaXY < -0.0002) )"


# FINAL CUT EXPRESSION:
cut_expression = f"({cut1}) & ({cut2}) & ({cut3})"

In [4]:
# CODE FOR UNIFYING DIRECTORIES OF A SINGLE ROOT FILE from all three type of T-Trees:

# Load T-Trees from directories
names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")
#num = 400   # number of directories to unify
num = len(names_coll)
# if I try with all, it crashes, because in this notebook I'm doing the cuts only after loading all data (in order tp study the cuts)

list_of_df = []               # add the dataframes in a list (we will concat them later)

collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

for i in range(num):          # cycle over directories (each with 3 TTrees)
    
    # Read collision tree
    df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")   # I take the fPosZ column as a DataFrame
    # list_of_collision_df.append(df_coll)               # add the dataframe in a list (we will concat them later)

    # # Read track and trackextr using boolean mask for track and trackextr:
    df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
         "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTPCpr", "fNsigmaTOFpi", "fNsigmaTOFka", "fNsigmaTOFpr"], library="pd" )
    df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
    mask = df_trackextr.eval(cut_expression)        # create boolean mask
    # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns):
    df_trackextr = df_trackextr.loc[mask, ["fPt", "fEta", "fCharge", "fDcaXY"] ].reset_index(drop=True)
    df_track = df_track.loc[mask,].reset_index(drop=True)
    # merging in a single dataframe
    df_trackextr["fIndexCollisions"] = df_track["fIndexCollisions"] 
    df_trackextr["fAlpha"] = df_track["fAlpha"]
    df_trackextr["fX"] = df_track["fX"]
    df_trackextr["fY"] = df_track["fY"]
    df_trackextr["fZ"] = df_track["fZ"]

    # we cut rows where the fIndexCollision is negative (for some reason)
    valid = df_track["fIndexCollisions"] >= 0
    df_track = df_track[valid].reset_index(drop=True)          
    df_trackextr = df_trackextr[valid].reset_index(drop=True)  

    
    # Now we for correct fPosZ (and add that column)
    df_trackextr["fPosZ"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosZ"].values
  
    df_trackextr["fPosX"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosX"].values

    df_trackextr["fPosY"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosY"].values

    df_trackextr = df_trackextr[(df_trackextr["fPosZ"] < 10) & (df_trackextr["fPosZ"] > -10)].reset_index(drop=True)
    df_trackextr["fIndexCollisions"] += collision_offset   # Fix local fIndexCollisions → global index 

    # save results:
    # ALTERNATIVE 1: for the first cycle, let's copy the first dataframe, then we concatenate the next ones
    if  i==0: df = df_trackextr
    else: df = pd.concat([df, df_trackextr], ignore_index=True)
    # # ALTERNATIVE 2:
    # list_of_df.append( df_trackextr )                  # add the dataframe in a list (we will concat them later)
    
    # Update offset for next loop
    collision_offset += len(df_coll)     

    # let's free the memory RAM of unused dataframes:
    del df_trackextr
    del df_track
    gc.collect()


# # UNCOMMENT FOR ALTERNATIVE 2:
# df = pd.concat(list_of_df, ignore_index=True)

# Merge everything in the total dataframe
N = len(df)


In [5]:
# debug
print("The dataframe has", len(df), "rows")
memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df

The dataframe has 4878200 rows
The dataframe occupy 223.31 MB


,fPt,fEta,fCharge,fDcaXY,fIndexCollisions,fAlpha,fX,fY,fZ,fPosZ,fPosX,fPosY
0,0.663068,0.598317,1,0.004326,0,-0.322267,-0.017226,-0.030717,-4.129672,-4.128410,-0.027438,-0.027783
1,1.434872,0.156435,-1,0.001984,2,-0.783967,-0.007016,-0.037420,-8.385111,-8.386139,-0.032792,-0.022949
2,1.418317,-0.302028,-1,-0.000906,3,1.478421,-0.035237,0.028537,4.196383,4.198601,-0.032568,-0.032371
3,0.629268,-0.664176,-1,-0.001437,3,1.949002,-0.018057,0.040782,4.195937,4.198601,-0.032568,-0.032371
4,1.047158,-0.719631,1,-0.006082,3,-0.223422,-0.024587,-0.044864,4.194757,4.198601,-0.032568,-0.032371
...,...,...,...,...,...,...,...,...,...,...,...,...
4878195,0.521796,0.090340,-1,0.004239,1666064,-1.776574,0.034615,-0.015266,-6.391246,-6.388824,-0.026167,-0.029899
4878196,0.924928,-0.743287,-1,-0.008164,1666064,2.492877,0.002787,0.031470,-6.380364,-6.388824,-0.026167,-0.029899
4878197,0.638627,-0.544448,-1,-0.000368,1666065,-2.320294,0.047770,-0.004037,-4.530087,-4.533272,-0.035230,-0.032469
4878198,0.915340,0.540868,1,-0.005042,1666065,1.709269,-0.027295,0.034333,-4.541160,-4.533272,-0.035230,-0.032469


In [6]:
# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484

# moment columns:
df["px"] = df["fPt"] * np.cos(df["fAlpha"])
df["py"] = df["fPt"] * np.sin(df["fAlpha"])
df["pz"] = df["fPt"] * np.sinh(df["fEta"])

# energy column (differentiating pions and kaons):
mass = np.where(df["fCharge"] > 0, m_pi, m_K)
df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)

# Debug
print("The dataframe has", len(df), "rows")
memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df

The dataframe has 4878200 rows
The dataframe occupy 316.35 MB


,fPt,fEta,fCharge,fDcaXY,fIndexCollisions,fAlpha,fX,fY,fZ,fPosZ,fPosX,fPosY,px,py,pz,Ene
0,0.663068,0.598317,1,0.004326,0,-0.322267,-0.017226,-0.030717,-4.129672,-4.128410,-0.027438,-0.027783,0.628933,-0.210006,0.420822,0.797640
1,1.434872,0.156435,-1,0.001984,2,-0.783967,-0.007016,-0.037420,-8.385111,-8.386139,-0.032792,-0.022949,1.016059,-1.013154,0.225381,1.534070
2,1.418317,-0.302028,-1,-0.000906,3,1.478421,-0.035237,0.028537,4.196383,4.198601,-0.032568,-0.032371,0.130831,1.412270,-0.434913,1.563487
3,0.629268,-0.664176,-1,-0.001437,3,1.949002,-0.018057,0.040782,4.195937,4.198601,-0.032568,-0.032371,-0.232359,0.584797,-0.449357,0.917397
4,1.047158,-0.719631,1,-0.006082,3,-0.223422,-0.024587,-0.044864,4.194757,4.198601,-0.032568,-0.032371,1.021130,-0.232017,-0.820313,1.337510
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4878195,0.521796,0.090340,-1,0.004239,1666064,-1.776574,0.034615,-0.015266,-6.391246,-6.388824,-0.026167,-0.029899,-0.106618,-0.510787,0.047203,0.719872
4878196,0.924928,-0.743287,-1,-0.008164,1666064,2.492877,0.002787,0.031470,-6.380364,-6.388824,-0.026167,-0.029899,-0.737039,0.558808,-0.752563,1.290566
4878197,0.638627,-0.544448,-1,-0.000368,1666065,-2.320294,0.047770,-0.004037,-4.530087,-4.533272,-0.035230,-0.032469,-0.435078,-0.467495,-0.365133,0.885936
4878198,0.915340,0.540868,1,-0.005042,1666065,1.709269,-0.027295,0.034333,-4.541160,-4.533272,-0.035230,-0.032469,-0.126345,0.906579,0.519572,1.061735


In [7]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * x_SV + row1["fZ"]
    z2_track = pz2/px2 * x_SV + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [10]:
# ALTERNATIVE 3:
# let's initialize some lists, then we will create a dataframe
collision_indices = []
track1_indices = []
track2_indices = []
dcaXY_products = []
inv_masses = []
inv_masses_approx = []
pt_totals = []
pz_totals = []
SV_X = []
SV_Y = []
SV_Z = []
decay_lengths = []
cos_pointings = []

i=0 # debug variable

# let's divide the dataframe for positive and negative charged
df_pos = df[ df['fCharge']>0 ]
df_neg = df[ df['fCharge']<0 ]

max_iters=10_000


# Iterate over each collision group
for collision_idx in ((df_neg['fIndexCollisions'].unique())[:max_iters]):
    group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
    group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]

    # Only collisions with at least a pair
    if len(group_pos) < 1:   continue

    # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
    group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
    group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})

    # let's crate indexes for all possible pairs:
    combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )

    # Iterate over all unique pairs of tracks
    for combo in combinat:
        row_neg = group_neg.iloc[combo[0]]
        row_pos = group_pos.iloc[combo[1]]

        product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
        
        # INVARIANT MASS calculation
        pt1, pt2 = row_neg['fPt'], row_pos['fPt']
        # eta1, eta2 = row_neg['fEta'], row_pos['fEta']
        # phi1, phi2 = row_neg['fAlpha'], row_pos['fAlpha']
        # delta_eta = eta1 - eta2
        # delta_phi = phi1 - phi2

        # approximation formula
        # inv_mass_approx = np.sqrt(2 * pt1 * pt2 * (np.cosh(delta_eta) - np.cos(delta_phi)))

        # exact formula:
        E1 = row_neg['Ene']
        E2 = row_pos['Ene']
        px1 = row_neg["px"]
        py1 = row_neg["py"]
        pz1 = row_neg["pz"]
        px2 = row_pos["px"]
        py2 = row_pos["py"]
        pz2 = row_pos["pz"]
        inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )


        # total transverse momentum of the D0 candidate (used later for sliced plots)
        pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)

        # secondary vertex
        SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
        SV_X.append(SV_coords[0])
        SV_Y.append(SV_coords[1])
        SV_Z.append(SV_coords[2])

        # decay length: distance between PV and SV
        PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
        decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )

        # cosine of pointing angle: the latter is the angle between the direction of the mother particle and the line connecting PV and SV
        mother_direction = [px1+px2, py1+py2, pz1+pz2]
        flight_line = SV_coords - PV_coords
        cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )
        
        # let's add the found pairs to the lists
        collision_indices.append(int(row_neg['fIndexCollisions']))
        # track1_indices.append(int(row_neg['orig_index']))
        # track2_indices.append(int(row_pos['orig_index']))
        dcaXY_products.append(product_dcaXY)
        inv_masses.append(inv_mass)
        # inv_masses_approx.append(inv_mass_approx)
        pt_totals.append(pt_total)
        pz_totals.append(pz1+pz2)

    # # let's free the memory RAM of unused dataframes:
    # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # del group
    # gc.collect()

    # debug:
    i += 1
    if i % 10000 ==0: print(i, end=' ')


# create a dataframe with the result:
df_pairs = pd.DataFrame({
    'collision_index': collision_indices,
    # 'track1_index': track1_indices,
    # 'track2_index': track2_indices,
    'dcaXY_product': dcaXY_products,
    'inv_mass': inv_masses,
    # 'inv_mass_approx': inv_masses_approx,
    'pt': pt_totals,
    'pz': pz_totals,
    # 'X_SV': SV_X,
    # 'Y_SV': SV_Y,
    # 'Z_SV': SV_Z,
    'decay_length': decay_lengths,
    'cos_pointing': cos_pointings
})

In [11]:
# Debug
print("The dataframe has", len(df_pairs), "rows")
memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"The dataframe occupy {memory:.2f} MB")
df_pairs

The dataframe has 80143 rows
The dataframe occupy 4.28 MB


,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,3,5.512908e-06,2.029822,1.649246,-1.255227,0.063825,-0.656131
1,3,-8.264114e-07,1.177901,2.772398,-0.547610,0.052611,-0.136851
2,3,7.114767e-06,0.837150,1.725034,-0.601303,0.062872,-0.479575
3,3,-5.188740e-07,2.151625,0.695278,-0.862479,0.003954,0.361597
4,3,4.383081e-06,1.363588,3.236413,-0.808830,0.059079,-0.336294
...,...,...,...,...,...,...,...
80138,25291,-7.955186e-04,2.020209,1.158146,-0.043845,0.593315,-0.526521
80139,25291,-2.809417e-04,1.236061,1.603832,-0.271161,2.323997,0.094560
80140,25291,-1.192836e-06,1.237885,1.251180,-0.829415,0.054858,-0.563974
80141,25291,4.925529e-06,1.152732,1.343368,-0.567813,0.016709,-0.453444


In [26]:
save_name = "pairs_" + "_".join([which_chunk, which_number])
save_name

'pairs_Chunk0950_001_007'

In [29]:
df_pairs.to_pickle(save_name + ".pkl")

In [30]:
# test = pd.read_pickle(save_name + ".pkl")
# test

,collision_index,dcaXY_product,inv_mass,pt,pz,decay_length,cos_pointing
0,3,5.512908e-06,2.029822,1.649246,-1.255227,0.063825,-0.656131
1,3,-8.264114e-07,1.177901,2.772398,-0.547610,0.052611,-0.136851
2,3,7.114767e-06,0.837150,1.725034,-0.601303,0.062872,-0.479575
3,3,-5.188740e-07,2.151625,0.695278,-0.862479,0.003954,0.361597
4,3,4.383081e-06,1.363588,3.236413,-0.808830,0.059079,-0.336294
...,...,...,...,...,...,...,...
80138,25291,-7.955186e-04,2.020209,1.158146,-0.043845,0.593315,-0.526521
80139,25291,-2.809417e-04,1.236061,1.603832,-0.271161,2.323997,0.094560
80140,25291,-1.192836e-06,1.237885,1.251180,-0.829415,0.054858,-0.563974
80141,25291,4.925529e-06,1.152732,1.343368,-0.567813,0.016709,-0.453444
